In [7]:
github_workflow_yaml = """
name: Security Scan (SonarQube + OWASP ZAP)

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ '*' ]

permissions:
  contents: read
  actions: write
  id-token: write

# NOTE:
# - Provide these secrets in your repo settings:
#   - SONAR_TOKEN : a SonarQube token (for self-hosted SonarQube or SonarCloud token)
#   - SONAR_PROJECT_KEY : unique key for your project (e.g. myorg_myrepo)
#   - ZAP_TARGET_URL : URL to scan with ZAP (e.g. https://staging.example.com). If your app is built and run in the job, use http://localhost:8080
#   - (Optional) SONAR_HOST_URL defaults to http://localhost:9000 for the local Sonar service below, override if using SonarCloud.
#
# If you use SonarCloud instead of a local SonarQube service:
# - set SONAR_HOST_URL=https://sonarcloud.io
# - provide SONAR_TOKEN and SONAR_ORGANIZATION (and adapt sonar.projectKey as required).

jobs:
  security-scan:
    runs-on: ubuntu-latest
    services:
      sonarqube:
        image: sonarqube:9.9-community
        ports:
          - 9000:9000
        options: >-
          --health-cmd="curl -f http://localhost:9000 || exit 1"
          --health-interval=10s
          --health-timeout=5s
          --health-retries=30
        env:
          - SONAR_JAVA_OPTS=-Xmx512m
    env:
      SONAR_HOST_URL: ${{ secrets.SONAR_HOST_URL || 'http://localhost:9000' }}
      SONAR_TOKEN: ${{ secrets.SONAR_TOKEN }}
      SONAR_PROJECT_KEY: ${{ secrets.SONAR_PROJECT_KEY || 'myorg_myrepo' }}
      ZAP_TARGET_URL: ${{ secrets.ZAP_TARGET_URL || 'http://localhost:8080' }}
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Set up JDK (required by some builds / sonar scanner)
        uses: actions/setup-java@v4
        with:
          distribution: temurin
          java-version: '17'

      - name: Wait for SonarQube to be ready
        run: |
          echo "Waiting for SonarQube at $SONAR_HOST_URL ..."
          for i in {1..60}; do
            if curl -sSf "$SONAR_HOST_URL" >/dev/null 2>&1; then
              echo "SonarQube is reachable"
              break
            fi
            echo -n "."
            sleep 5
          done
          # show server info for debugging
          curl -sSf "$SONAR_HOST_URL/api/server/version" || true

      - name: Install sonar-scanner CLI
        run: |
          # Download sonar-scanner CLI (Linux)
          SC_VER=4.10.0.6089
          wget -qO - https://binaries.sonarsource.com/Distribution/sonar-scanner-cli/sonar-scanner-cli-${SC_VER}.zip -O scanner.zip
          unzip -q scanner -d scanner
          echo "SONAR_SCANNER_HOME=$(pwd)/scanner/sonar-scanner-${SC_VER}" >> $GITHUB_ENV
          echo "::add-path::${SONAR_SCANNER_HOME}/bin"
        shell: bash

      - name: Build project (adjust this to your build tool)
        run: |
          # Example: Java/Gradle. Replace as needed:
          if [ -f "gradlew" ]; then
            ./gradlew clean assemble -x test
          elif [ -f "pom.xml" ]; then
            mvn -B -DskipTests package
          elif [ -f "package.json" ]; then
            npm ci
            npm run build --if-present
          else
            echo "No recognized build file found; skipping build step"
          fi

      - name: Run SonarQube scan
        env:
          SONAR_HOST_URL: ${{ env.SONAR_HOST_URL }}
          SONAR_TOKEN: ${{ env.SONAR_TOKEN }}
          SONAR_PROJECT_KEY: ${{ env.SONAR_PROJECT_KEY }}
        run: |
          set -e
          echo "Running sonar-scanner..."
          # Create sonar-project.properties with common settings; you may prefer per-project file
          cat > sonar-project.properties <<EOF
          sonar.projectKey=${SONAR_PROJECT_KEY}
          sonar.sources=.
          sonar.host.url=${SONAR_HOST_URL}
          sonar.login=${SONAR_TOKEN}
          sonar.java.binaries=build
          # set any language-specific properties here
EOF
          ${SONAR_SCANNER_HOME}/bin/sonar-scanner \
            -Dsonar.projectKey="${SONAR_PROJECT_KEY}" \
            -Dsonar.host.url="${SONAR_HOST_URL}" \
            -Dsonar.login="${SONAR_TOKEN}" \
            || echo "Sonar scanner finished with non-zero exit (see server for details)"

      - name: Wait for SonarQube analysis to be indexed
        env:
          SONAR_HOST_URL: ${{ env.SONAR_HOST_URL }}
          SONAR_TOKEN: ${{ env.SONAR_TOKEN }}
          SONAR_PROJECT_KEY: ${{ env.SONAR_PROJECT_KEY }}
        run: |
          echo "Polling SonarQube for analysis report..."
          for i in {1..40}; do
            resp=$(curl -s -u "${SONAR_TOKEN}:" "${SONAR_HOST_URL}/api/ce/component?component=${SONAR_PROJECT_KEY}")
            taskId=$(echo "$resp" | jq -r '.queue[0].taskId // .current?.id // empty')
            # fallback: query latest task in background
            if [ -z "$taskId" ]; then
              taskId=$(curl -s -u "${SONAR_TOKEN}:" "${SONAR_HOST_URL}/api/ce/activity?component=${SONAR_PROJECT_KEY}" | jq -r '.current[0].id // empty')
            fi
            if [ -n "$taskId" ]; then
              status=$(curl -s -u "${SONAR_TOKEN}:" "${SONAR_HOST_URL}/api/ce/task?id=${taskId}" | jq -r '.task.status // empty')
              echo "Task $taskId status: $status"
              if [ "$status" = "SUCCESS" ] || [ "$status" = "FAILED" ]; then
                break
              fi
            fi
            sleep 4
          done

      - name: Export SonarQube issues to JSON
        env:
          SONAR_HOST_URL: ${{ env.SONAR_HOST_URL }}
          SONAR_TOKEN: ${{ env.SONAR_TOKEN }}
          SONAR_PROJECT_KEY: ${{ env.SONAR_PROJECT_KEY }}
        run: |
          mkdir -p reports
          echo "Querying SonarQube issues for project ${SONAR_PROJECT_KEY}..."
          # This queries SonarQube issues API paginated; returning first 5000 issues if present
          curl -s -u "${SONAR_TOKEN}:" "${SONAR_HOST_URL}/api/issues/search?componentKeys=${SONAR_PROJECT_KEY}&ps=5000" -o reports/sonar-issues.json
          echo "Saved Sonar issues to reports/sonar-issues.json"
          # Create a quick markdown summary
          jq -r '[.total, (.issues|length)] | "total_issues: \(.[])"' reports/sonar-issues.json > reports/sonar-summary.txt || true

      - name: Run OWASP ZAP baseline scan (docker)
        # This runs the ZAP baseline scan against ZAP_TARGET_URL and produces HTML + JSON reports
        run: |
          mkdir -p reports
          echo "Running OWASP ZAP baseline scan against: $ZAP_TARGET_URL"
          docker run --rm -v "$(pwd)/reports":/zap/reports owasp/zap2docker-stable \
            zap-baseline.py -t "$ZAP_TARGET_URL" -r /zap/reports/zap-report.html -J /zap/reports/zap-report.json -I -T 120 \
            || echo "ZAP baseline finished (exit code ignored to continue report collection)"
          ls -lah reports

      - name: Convert ZAP JSON to short markdown (quick human-readable)
        run: |
          python3 - <<'PY'
import json,sys
try:
    j=json.load(open('reports/zap-report.json'))
except Exception as e:
    print("No ZAP JSON present or failed to parse:",e); sys.exit(0)
alerts=j.get('site',[])
lines=[]
for site in alerts:
    site_name=site.get('@name','<site>')
    for alert in site.get('alerts',[]):
        risk=alert.get('risk')
        name=alert.get('alert')
        desc=alert.get('desc','').split('\n')[0][:200]
        lines.append(f"- [{risk}] {name} — {desc}")
open('reports/zap-summary.md','w').write("# ZAP Findings\n\n" + "\n".join(lines))
print("zap-summary.md written")
PY

      - name: Upload reports as workflow artifacts
        uses: actions/upload-artifact@v4
        with:
          name: security-scan-reports
          path: reports/**/*

      - name: Print small summary to job log
        run: |
          echo "=== Quick summary ==="
          echo "Sonar issues (first 200 chars):"
          head -c 200 reports/sonar-issues.json || true
          echo
          echo "ZAP summary (first 200 chars):"
          head -c 200 reports/zap-summary.md || true
"""

# Now the `github_workflow_yaml` variable holds the YAML content as a string.
# You can print it to verify:
# print(github_workflow_yaml)

# Or write it to a file, for example:
# with open('your-workflow.yml', 'w') as f:
#     f.write(github_workflow_yaml)


<>:153: SyntaxWarning: invalid escape sequence '\('
<>:153: SyntaxWarning: invalid escape sequence '\('
/tmp/ipython-input-1580774550.py:153: SyntaxWarning: invalid escape sequence '\('
  jq -r '[.total, (.issues|length)] | "total_issues: \(.[])"' reports/sonar-issues.json > reports/sonar-summary.txt || true
